# S0 - auditoria amostral e agregada do corpus

Esta etapa separa duas evidências. A auditoria amostral é um smoke test limitado; o perfil agregado consulta o Parquet completo com DuckDB e não materializa narrativas em Python.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'dataset' / 'processed' / 'complaints.parquet').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find project root')
    PROJECT_ROOT = parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from time import perf_counter

import polars as pl

from consumer_complaint_intelligence.audit import audit_s0_corpus
from consumer_complaint_intelligence.audit import audit_s0_sample
from consumer_complaint_intelligence.config import ProjectPaths
from consumer_complaint_intelligence.config import S0AuditConfig

paths = ProjectPaths.from_root(PROJECT_ROOT)
audit_config = S0AuditConfig(sample_rows=100_000, top_k=15)


## Evidência amostral

`audit_s0_sample` lê no máximo 100.000 linhas pelo início do Parquet. Seus percentuais e duplicatas normalizadas descrevem somente essa amostra, não o corpus.


In [2]:
sample_started = perf_counter()
sample_report = audit_s0_sample(
    paths.require_parquet(), config=audit_config
)
sample_elapsed_seconds = round(perf_counter() - sample_started, 3)
sample_summary = {
    'evidence': sample_report['scope']['evidence'],
    'rows_scanned': sample_report['scope']['rows_scanned'],
    'narrative_coverage_pct': sample_report[
        'narrative_coverage'
    ]['narrative_coverage_pct'],
    'total_duplicate_groups_in_sample': sample_report['duplicates'][
        'total_duplicate_groups'
    ],
    'duration_seconds': sample_elapsed_seconds,
}
sample_summary


{'evidence': 'sample_only',
 'rows_scanned': 100000,
 'narrative_coverage_pct': 1.453,
 'total_duplicate_groups_in_sample': 26,
 'duration_seconds': 0.233}

## Evidência do corpus completo

`audit_s0_corpus` executa consultas agregadas separadas, com limite de memória, dois threads e spill em `temp/duckdb`. O texto das narrativas permanece dentro do DuckDB.


In [3]:
full_started = perf_counter()
full_report = audit_s0_corpus(
    paths.require_parquet(),
    temp_directory=paths.temp_dir / 'duckdb',
    config=audit_config,
)
full_elapsed_seconds = round(perf_counter() - full_started, 3)
full_summary = {
    'evidence': full_report['scope']['evidence'],
    'total_rows': full_report['volume']['total_rows'],
    'narrative_rows': full_report['volume']['narrative_rows'],
    'narrative_coverage_pct': full_report['volume'][
        'narrative_coverage_pct'
    ],
    'duration_seconds': full_elapsed_seconds,
    'query_timings_seconds': full_report['query_timings_seconds'],
}
full_summary


{'evidence': 'complete_parquet_corpus',
 'total_rows': 17094898,
 'narrative_rows': 3836659,
 'narrative_coverage_pct': 22.4433,
 'duration_seconds': 52.683,
 'query_timings_seconds': {'volume': 22.696,
  'coverage_by_year': 9.864,
  'product_by_year_label': 0.552,
  'issue_by_year_label': 0.844,
  'complaint_id': 3.713,
  'exact_narrative_duplicates': 15.01}}

## Cobertura, taxonomia e duplicatas

As tabelas abaixo são agregados do corpus completo. Duplicatas exatas usam MD5 mais tamanho do texto; a colisão é teoricamente possível e está registrada no relatório. Duplicatas normalizadas continuam disponíveis apenas na auditoria amostral.


In [4]:
taxonomy_summary = pl.DataFrame([
    {
        'field': 'Product',
        'total_rows': full_report['taxonomy']['product']['total_rows'],
        'distinct_labels': full_report['taxonomy']['product'][
            'distinct_labels'
        ],
        'missing_label_rows': full_report['taxonomy']['product'][
            'missing_label_rows'
        ],
        'first_year': full_report['taxonomy']['product']['first_year'],
        'last_year': full_report['taxonomy']['product']['last_year'],
        'number_years': full_report['taxonomy']['product'][
            'number_years'
        ],
    },
    {
        'field': 'Issue',
        'total_rows': full_report['taxonomy']['issue']['total_rows'],
        'distinct_labels': full_report['taxonomy']['issue'][
            'distinct_labels'
        ],
        'missing_label_rows': full_report['taxonomy']['issue'][
            'missing_label_rows'
        ],
        'first_year': full_report['taxonomy']['issue']['first_year'],
        'last_year': full_report['taxonomy']['issue']['last_year'],
        'number_years': full_report['taxonomy']['issue'][
            'number_years'
        ],
    },
])
display(taxonomy_summary)
coverage_by_year = pl.DataFrame(
    full_report['volume']['coverage_by_year']
)
product_by_year_label = pl.DataFrame(
    full_report['taxonomy']['product']['counts_by_year_label']
)
issue_by_year_label = pl.DataFrame(
    full_report['taxonomy']['issue']['counts_by_year_label']
)
display(coverage_by_year)
display(product_by_year_label)
display(issue_by_year_label)
display(full_report['complaint_id'])
display(full_report['exact_narrative_duplicates'])


field,total_rows,distinct_labels,missing_label_rows,first_year,last_year,number_years
str,i64,i64,i64,i64,i64,i64
"""Product""",17094898,21,0,2011,2026,16
"""Issue""",17094898,178,6,2011,2026,16


year,total_rows,narrative_rows,narrative_coverage_pct
i64,i64,i64,f64
2011,2536,0,0.0
2012,72368,0,0.0
2013,108214,0,0.0
2014,152909,0,0.0
2015,168273,54723,32.5204
…,…,…,…
2022,800245,337273,42.1462
2023,1292049,487410,37.7238
2024,2734269,814384,29.7843


year,label,rows,is_missing
i64,str,i64,bool
2011,"""Credit card""",1260,false
2011,"""Mortgage""",1276,false
2012,"""Bank account or service""",12210,false
2012,"""Consumer Loan""",1986,false
2012,"""Credit card""",15352,false
…,…,…,…
2026,"""Mortgage""",20231,false
2026,"""Payday loan, title loan, perso…",10923,false
2026,"""Prepaid card""",3602,false


year,label,rows,is_missing
i64,str,i64,bool
2011,"""APR or interest rate""",121,false
2011,"""Advertising and marketing""",26,false
2011,"""Application processing delay""",9,false
2011,"""Application, originator, mortg…",151,false
2011,"""Arbitration""",2,false
…,…,…,…
2026,"""Vehicle was repossessed or sol…",95,false
2026,"""Was approved for a loan, but d…",14,false
2026,"""Was approved for a loan, but d…",61,false


{'total_rows': 17094898,
 'distinct_non_null_ids': 17094898,
 'null_id_rows': 0,
 'redundant_non_null_id_rows': 0,
 'is_unique_non_null': True}

{'method': 'md5_plus_text_length_inside_duckdb',
 'total_duplicate_groups': 246096,
 'duplicate_group_rows': 1499532,
 'redundant_rows': 1253436,
 'narratives_materialized_in_python': False,
 'collision_risk': 'Counts are keyed by MD5 and text length. MD5 collisions are theoretical but not impossible; confirm critical groups with a collision-safe follow-up before publication.'}

A S0 ainda não consolida renomes históricos, não define o split temporal final e não treina modelo.
